Environment Variable Setup

In [ ]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

Data Preparation

In [ ]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=128, 
                                               chunk_overlap=16,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=True,)
chunks = text_splitter.split_documents(docs)

Build Chain

In [ ]:
import time
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough,RunnableLambda
from langchain.schema.output_parser import StrOutputParser

vector_storage = FAISS.from_documents(chunks, embedding)

# Set retriever
retriever = vector_storage.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
# Prompt template
template = """
You are a helpful assistant specializing in 3D printing operations.  
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | RunnableLambda(format_docs),  
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("What is extruder blob in 3D printing?")
response

Generate evalutaion test dataset by RAGAS

In [ ]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):
        # get retrieved docs
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]

        # Record response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times
        

Using LLM to evaluate dataset

In [ ]:
from ragas import evaluate
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        )
    
    return result


In [ ]:
# select metrics
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),
         FactualCorrectness(mode = 'f1', atomicity='high', coverage='high')]


eval_ds, response_time = get_eval_ds("evaluate_dataset/test_dataset.csv")

In [ ]:
result = get_eval_result(eval_ds, metrics)

In [ ]:
result

evaluate dataset without noise

In [ ]:
eval_ds_cor, response_time_cor = get_eval_ds("evaluate_dataset/test_dataset_corrected.csv")

In [ ]:
result_cor = get_eval_result(eval_ds_cor, metrics)

In [ ]:
result_cor

Save Results

In [ ]:
# save results
df = result.to_pandas()
df['response_time'] = response_time
df.to_csv("evaluate_results/01_naive_rag/test_dataset.csv")

df_cor = result_cor.to_pandas()
df_cor['response_time'] = response_time_cor
df_cor.to_csv("evaluate_results/01_naive_rag/test_dataset_cor.csv")